In [1]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

#손 인식하는 기준/ 옵션을 설정
hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

#캠 켜기
cap = cv2.VideoCapture(0)

print("카메라가 성공적으로 켜졌습니다.")
print("종료하고 싶으시다면 카메라 화면 창을 마우스로 클릭한 후, 키보드 'q'를 누르세요.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("카메라 화면을 불러올 수 없습니다.")
        break

    # 3. 이미지 전처리
    frame = cv2.flip(frame, 1)#좌우반전
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)    
    #손가락 특징점 추적
    results = hands.process(img_rgb)

    #특징점들, 연결선 그리기
    if results.multi_hand_features:
        for hand_features in results.multi_hand_features:
            mp_drawing.draw_features(
                frame, 
                hand_features, 
                mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=4),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
            )

    #결과를 화면에 나타냄
    cv2.imshow('Su-eo Project: Hand Tracking', frame)

    #화면창을 끄는 방법
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()

카메라가 성공적으로 켜졌습니다.
종료하고 싶으시다면 카메라 화면 창을 마우스로 클릭한 후, 키보드 'q'를 누르세요.


AttributeError: type object 'SolutionOutputs' has no attribute 'multi_hand_features'

In [2]:
import sys
print(sys.executable)

C:\Users\seou3\anaconda3\envs\sign312\python.exe


In [3]:
import platform
print(platform.python_version())

3.12.13


In [4]:
import mediapipe as mp

print(mp.__file__)
print(dir(mp))

C:\Users\seou3\anaconda3\envs\sign312\Lib\site-packages\mediapipe\__init__.py
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [ ]:
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.14

Found existing installation: mediapipe 0.10.35
Uninstalling mediapipe-0.10.35:
  Successfully uninstalled mediapipe-0.10.35


In [1]:
import mediapipe as mp

print(mp.__version__)
print(hasattr(mp, "solutions"))

ModuleNotFoundError: No module named 'mediapipe'

In [2]:
import sys
!{sys.executable} -m pip install mediapipe==0.10.14

  Using cached mediapipe-0.10.14-cp312-cp312-win_amd64.whl.metadata (9.9 kB)
  Using cached jax-0.10.1-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.1-cp312-cp312-win_amd64.whl.metadata (1.4 kB)
  Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached ml_dtypes-0.5.4-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
    --------------------------------------- 1.0/50.8 MB 5.6 MB/s eta 0:00:09
   - -------------------------------------- 2.1/50.8 MB 5.3 MB/s eta 0:00:10
   -- ------------------------------------- 2.9/50.8 MB 5.2 MB/s eta 0:00:10
   --- ------------------------------------ 4.2/50.8 MB 5.1 MB/s eta 0:00:10
   --- ------------------------------------ 5.0/50.8 MB 5.0 MB/s eta 0:00:10
   ---- ----------------------------------- 6

In [1]:
import mediapipe as mp

print(mp.__version__)
print(hasattr(mp, "solutions"))

0.10.14
True


In [1]:
import cv2
import mediapipe as mp
import time
import numpy as np
import torch

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)

sequence = []

if cap.isOpened():
    print("카메라가 성공적으로 켜졌습니다.", flush=True)
    print("종료하려면 카메라 창 클릭 후 q를 누르세요.", flush=True)
    time.sleep(2)
else:
    print("카메라 연결 실패")

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        print("카메라 화면을 불러올 수 없습니다.")
        break

    frame = cv2.flip(frame, 1)

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(img_rgb)
    frame_keypoints = []

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            for lm in hand_landmarks.landmark:
                frame_keypoints.append(lm.x)
                frame_keypoints.append(lm.y)
                frame_keypoints.append(1)

    while len(frame_keypoints) < 126:
        frame_keypoints.append(0)

    frame_keypoints = frame_keypoints[:126]

    extra_features = [0, 0, 0, 0, 0, 0]
    frame_keypoints = frame_keypoints + extra_features
    
    sequence.append(frame_keypoints)
    if len(sequence) == 30:

        sequence_array = np.array(sequence)
        sequence_tensor = torch.tensor(sequence_array, dtype=torch.float32)
        sequence_tensor = sequence_tensor.unsqueeze(0)

        print("30프레임 수집 완료")
        print("sequence shape:", sequence_array.shape)
        print("tensor shape:", sequence_tensor.shape)

        sequence = []


    print("좌표 개수:", len(frame_keypoints))
    print(frame_keypoints[:9])
    if results.multi_hand_landmarks:

        for hand_landmarks in results.multi_hand_landmarks:

            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=4),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
            )

    cv2.imshow("Su-eo Project: Hand Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

카메라가 성공적으로 켜졌습니다.
종료하려면 카메라 창 클릭 후 q를 누르세요.


C:\Users\seou3\anaconda3\envs\sign312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


좌표 개수: 132
[0.26784807443618774, 0.7617235779762268, 1, 0.29802653193473816, 0.8177833557128906, 1, 0.35169515013694763, 0.8333845138549805, 1]
좌표 개수: 132
[0.2604023814201355, 0.7635924220085144, 1, 0.29837676882743835, 0.8237649202346802, 1, 0.3551081717014313, 0.845793604850769, 1]
좌표 개수: 132
[0.2598525583744049, 0.7660817503929138, 1, 0.28843972086906433, 0.7744174599647522, 1, 0.3438621461391449, 0.7717288136482239, 1]
좌표 개수: 132
[0.2606932520866394, 0.7865921258926392, 1, 0.27922800183296204, 0.7200565934181213, 1, 0.3320460915565491, 0.6685030460357666, 1]
좌표 개수: 132
[0.25832489132881165, 0.8037962913513184, 1, 0.2767804265022278, 0.7308112382888794, 1, 0.3285623788833618, 0.6782215237617493, 1]
좌표 개수: 132
[0.24902118742465973, 0.8032975196838379, 1, 0.27794212102890015, 0.725299596786499, 1, 0.33396053314208984, 0.6835395097732544, 1]
좌표 개수: 132
[0.2398047149181366, 0.8123536109924316, 1, 0.26789891719818115, 0.7319336533546448, 1, 0.32117125391960144, 0.6895091533660889, 1]
좌표 

In [6]:
!pip install torch

   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   - -------------------------------------- 5.0/123.0 MB 25.8 MB/s eta 0:00:05
   ---- ----------------------------------- 12.8/123.0 MB 31.7 MB/s eta 0:00:04
   ------ --------------------------------- 19.1/123.0 MB 30.8 MB/s eta 0:00:04
   -------- ------------------------------- 26.2/123.0 MB 31.6 MB/s eta 0:00:04
   ----------- ---------------------------- 34.6/123.0 MB 33.2 MB/s eta 0:00:03
   ------------- -------------------------- 43.0/123.0 MB 34.3 MB/s eta 0:00:03
   ---------------- ----------------------- 51.6/123.0 MB 35.5 MB/s eta 0:00:03
   ----------------- ---------------------- 54.5/123.0 MB 32.8 MB/s eta 0:00:03
   ----------------- ---------------------- 54.8/123.0 MB 29.9 MB/s eta 0:00:03
   ------------------ --------------------- 57.9/123.0 MB 28.0 MB/s eta 0:00:03
   ------------------- -------------------- 60.8/123.0 MB 26.6 MB/s eta 0:00:03
   --------------------- ------------------ 66.1/1

In [1]:
import torch
print(torch.__version__)

ModuleNotFoundError: No module named 'torch'

In [4]:
!pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -------------------------------------- - 3.9/4.1 MB 22.9 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 19.0 MB/s  0:00:00

   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   -------------------- ------------------- 1/2 [torchvision]
   -------------------- ------------------- 1/2 [torchvision]
   -------------------- ------------------- 1/2 [torchvision]
   -------------------- ------------------- 1/2 [torchvision]
   -------------------- ------------------- 1/2 [torchvision]
   -------------------- ------------------

In [1]:
import torch
print(torch.__version__)

ModuleNotFoundError: No module named 'torch'

In [2]:
import sys
print(sys.executable)
!{sys.executable} -m pip install torch torchvision torchaudio

C:\Users\seou3\anaconda3\envs\sign312\python.exe
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   -- ------------------------------------- 6.3/123.0 MB 32.2 MB/s eta 0:00:04
   ----- ---------------------------------- 15.7/123.0 MB 38.1 MB/s eta 0:00:03
   ------ --------------------------------- 21.5/123.0 MB 34.9 MB/s eta 0:00:03
   --------- ------------------------------ 30.1/123.0 MB 36.1 MB/s eta 0:00:03
   ------------ --------------------------- 38.3/123.0 MB 36.3 MB/s eta 0:00:03
   --------------- ------------------------ 47.4/123.0 MB 37.7 MB/s eta 0:00:03
   ------------------ --------------------- 55.8/123.0 MB 37.8 MB/s eta 0:00:02
   -------------------- ------------------- 63.4/123.0 MB 37.4 MB/s eta 0:00:02
   ----------------------- ---------------- 72.1/123.0 MB 37.7 MB/s eta 0:00:02
   -------------------------- ------------- 82.1/123.0 MB 38.8 MB/s eta 0:00:02
   ------------------------------ --------- 92.3/123.0 MB 39.5 MB/s eta 0:00:01
 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import torch
print(torch.__version__)

2.12.0+cpu
